# Train the PCB check: "is there a PCB in this picture?"

**Project:** Bharat AI Hackathon, on-device PCB inspection on Raspberry Pi 5.

**Why:** the station must give a third answer, *no PCB in view*, instead of PASS or FAIL when the camera
is not looking at a board. This notebook trains a very small image classifier (YOLOv8n-cls) with two
classes, `pcb` and `not_pcb`, and gives you `pcb_gate.onnx` for the Pi.

**Pictures it learns from:**
- `pcb`: the 693 photos of the PKU PCB dataset, plus **your own camera photos** of PCBs and printed PCB sheets.
- `not_pcb`: 1,000 everyday photos from Imagenette (a free subset of ImageNet), plus **your own camera photos**
  of the empty stand, blank paper, text pages, hands and other objects.

Your own photos matter most, because they show exactly what the station's camera sees. They are scored
separately at the end.

**Before you start:**
1. In Google Drive, make a folder `pcb_gate_photos` with two folders inside: `pcb` and `not_pcb`.
   Put your camera photos in them (at least 20 in each, ideally 40 or more).
2. In Colab: **Runtime → Change runtime type → T4 GPU → Save**.

Run the cells from top to bottom with **Shift + Enter**. The whole notebook takes roughly 15 to 25 minutes.

## Step 1: Check that the GPU is on
This should print a table that mentions **Tesla T4** (or another GPU).

In [ ]:
!nvidia-smi

## Step 2: Install the libraries
- `ultralytics` contains YOLO.
- `kagglehub` downloads the PCB dataset from Kaggle.

In [ ]:
!pip install -q ultralytics kagglehub

## Step 3: Copy your own photos from Google Drive
Colab asks for permission to read your Drive. The cell counts your photos and stops if a folder has fewer than 20.

In [ ]:
import glob, os, shutil

DRIVE_FOLDER = "/content/drive/MyDrive/pcb_gate_photos"
IMAGE_TYPES = (".jpg", ".jpeg", ".png", ".bmp")

from google.colab import drive
drive.mount("/content/drive")

own = {}
for label in ("pcb", "not_pcb"):
    folder = os.path.join(DRIVE_FOLDER, label)
    own[label] = sorted(p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                        if p.lower().endswith(IMAGE_TYPES))
    print(f"Your '{label}' photos: {len(own[label])}")

for label, paths in own.items():
    if len(paths) < 20:
        raise SystemExit(f"Please add at least 20 photos to {DRIVE_FOLDER}/{label} and run this cell again.")

## Step 4: Download the PCB dataset and the everyday photos
- The PCB dataset is the same one used for the defect detector (about 2 GB).
- Imagenette is a free set of everyday photos (fish, dogs, churches, trucks and so on), about 330 MB.

In [ ]:
import random, tarfile, urllib.request
import kagglehub

pcb_dir = kagglehub.dataset_download("akhatova/pcb-defects")
dataset_pcb = sorted(p for p in glob.glob(os.path.join(pcb_dir, "**", "*.jpg"), recursive=True)
                     if "rotation" not in p.lower())
print("PCB dataset photos:", len(dataset_pcb))

IMAGENETTE_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"
if not os.path.isdir("/content/imagenette2-320"):
    urllib.request.urlretrieve(IMAGENETTE_URL, "/content/imagenette.tgz")
    with tarfile.open("/content/imagenette.tgz") as tar:
        tar.extractall("/content")
everyday = sorted(glob.glob("/content/imagenette2-320/**/*.JPEG", recursive=True))
everyday = random.Random(0).sample(everyday, min(1000, len(everyday)))
print("Everyday photos used:", len(everyday))

## Step 5: Build the training folders
What this cell does:
1. **Splits the pictures** into `train` (to learn from) and `val` (to test on). 25% of your own photos and
   15% of the others are kept for testing and never used for learning.
2. **Shrinks every picture** to at most 640 pixels on its longest side, to make training faster. The model
   itself looks at 224 x 224 pixels.
3. **Repeats your own training photos three times**, because there are far fewer of them and they matter most.

YOLO's classifier expects one folder per class: `train/pcb`, `train/not_pcb`, `val/pcb`, `val/not_pcb`.

In [ ]:
from PIL import Image, ImageOps

DATA = "/content/gate_data"
OWN_REPEATS = 3
shutil.rmtree(DATA, ignore_errors=True)
for split in ("train", "val"):
    for label in ("pcb", "not_pcb"):
        os.makedirs(os.path.join(DATA, split, label))


def save_small(src, dst):
    img = ImageOps.exif_transpose(Image.open(src)).convert("RGB")   # respect phone rotation, drop transparency
    img.thumbnail((640, 640))
    img.save(dst, quality=90)


def split_list(paths, val_fraction, seed):
    paths = sorted(paths)
    random.Random(seed).shuffle(paths)
    n_val = max(1, round(val_fraction * len(paths)))
    return paths[n_val:], paths[:n_val]


own_val = {}                       # your own test photos, scored separately in Step 8
counts = {}
for label, others in (("pcb", dataset_pcb), ("not_pcb", everyday)):
    own_train, own_val[label] = split_list(own[label], 0.25, seed=1)
    other_train, other_val = split_list(others, 0.15, seed=2)
    n = {"train": 0, "val": 0}
    for i, src in enumerate(other_train):
        save_small(src, f"{DATA}/train/{label}/other_{i:05d}.jpg"); n["train"] += 1
    for i, src in enumerate(other_val):
        save_small(src, f"{DATA}/val/{label}/other_{i:05d}.jpg"); n["val"] += 1
    for i, src in enumerate(own_train):
        for r in range(OWN_REPEATS):
            save_small(src, f"{DATA}/train/{label}/own_{i:04d}_r{r}.jpg"); n["train"] += 1
    for i, src in enumerate(own_val[label]):
        save_small(src, f"{DATA}/val/{label}/own_{i:04d}.jpg"); n["val"] += 1
    counts[label] = n
    print(f"{label:8s}: train {n['train']:5d} pictures (your photos x{OWN_REPEATS}: {len(own_train)}), "
          f"val {n['val']:4d} pictures (your photos: {len(own_val[label])})")

## Step 6: Look at a few training pictures
Top row should all be PCBs, bottom row should all be not PCBs.

In [ ]:
from IPython.display import display

sheet = Image.new("RGB", (6 * 200, 2 * 200), "white")
for row, label in enumerate(("pcb", "not_pcb")):
    files = sorted(glob.glob(f"{DATA}/train/{label}/*.jpg"))
    own_files = [f for f in files if "/own_" in f]
    picks = random.Random(3).sample(own_files, min(3, len(own_files)))
    picks += random.Random(4).sample(files, 6 - len(picks))
    for col, path in enumerate(picks):
        tile = Image.open(path).convert("RGB"); tile.thumbnail((196, 196))
        sheet.paste(tile, (col * 200 + 2, row * 200 + 2))
display(sheet)

## Step 7: Train
- `yolov8n-cls.pt` is the smallest YOLO classifier, already trained on ImageNet. We fine-tune it for our two classes.
- `epochs=20`, `patience=8`: up to 20 passes, stopping early if it stops improving.

This takes roughly 5 to 15 minutes. The `top1_acc` column (accuracy on the test pictures) should get close to 1.0.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")
model.train(data=DATA, epochs=20, patience=8, imgsz=224, batch=64, device=0,
            project="/content/runs", name="pcb_gate", exist_ok=True)
save_dir = str(model.trainer.save_dir)
best = YOLO(os.path.join(save_dir, "weights", "best.pt"))

## Step 8: Score it on your own test photos
This is the number that matters: your own camera photos that were held back from training. The Pi treats a
picture as a PCB when the probability of `pcb` is at least 0.60, so the same rule is used here. Any mistakes
are listed by file name so you can look at them.

In [ ]:
THRESHOLD = 0.60

results = {}
mistakes = []
for label, paths in own_val.items():
    correct = 0
    for path in paths:
        r = best(path, imgsz=224, verbose=False)[0]
        pcb_index = [i for i, name in r.names.items() if name == "pcb"][0]
        p_pcb = float(r.probs.data[pcb_index])
        said = "pcb" if p_pcb >= THRESHOLD else "not_pcb"
        if said == label:
            correct += 1
        else:
            mistakes.append(f"{os.path.basename(path)}: is {label}, model said {said} (p_pcb = {p_pcb:.2f})")
    results[label] = (correct, len(paths))

all_val = best.val(data=DATA, imgsz=224, verbose=False)
summary = (
    "PCB check (YOLOv8n-cls, 224 x 224), threshold p(pcb) >= 0.60\n"
    f"Accuracy on all test pictures: {all_val.top1:.3f}\n"
    f"Your own PCB photos recognised as PCB: {results['pcb'][0]}/{results['pcb'][1]}\n"
    f"Your own non-PCB photos recognised as not PCB: {results['not_pcb'][0]}/{results['not_pcb'][1]}\n"
)
print(summary)
print("Mistakes:" if mistakes else "No mistakes on your own test photos.")
for m in mistakes:
    print("  " + m)

## Step 9: Convert for the Raspberry Pi and download
The zip holds `pcb_gate.onnx` for the Pi, `pcb_gate.pt` (keep it for re-training) and `gate_metrics.txt`.

In [ ]:
from google.colab import files

onnx_path = best.export(format="onnx", imgsz=224)
out = "/content/pcb_gate"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out)
shutil.copy(onnx_path, os.path.join(out, "pcb_gate.onnx"))
shutil.copy(os.path.join(save_dir, "weights", "best.pt"), os.path.join(out, "pcb_gate.pt"))
with open(os.path.join(out, "gate_metrics.txt"), "w") as f:
    f.write(summary + "\n" + "\n".join(mistakes) + "\n")
shutil.make_archive(out, "zip", out)
files.download(out + ".zip")